## Transfer Learning Approach

In this project, I applied **transfer learning** to adapt a pretrained convolutional neural network for a custom binary image classification task. Instead of training a model from scratch, I fine-tuned a pretrained model to distinguish between **authorized (Bo)** and **unauthorized (not-Bo)** images.

Transfer learning is particularly effective when working with limited data, as it allows the model to reuse learned visual features such as edges, textures, and shapes. This approach reduces training time, improves generalization, and minimizes overfitting compared to training a deep network from scratch.

## Model Architecture & Setup

I used a pretrained **VGG16** convolutional neural network as the base model and modified the final classification layers to match the binary classification task. Most of the base layers were frozen to preserve learned features, while the final layers were fine-tuned on the custom dataset.

Training was performed using **GPU acceleration (CUDA)** to speed up computation. Due to the relatively small dataset size, the model was trained for only a few epochs to reduce the risk of overfitting.

In [1]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms
import torchvision.io as tv_io

import glob
import json
from PIL import Image

import utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

True

## Problem Definition: Personalized Image Classification

The goal of this project was to build a **binary image classification system** capable of distinguishing between a specific authorized subject (“Bo”) and all other dogs.

This scenario mimics a real-world access control problem where a system must correctly identify a known entity while rejecting visually similar but unauthorized inputs. The primary challenge was the **limited dataset size** — only ~30 labeled images of the authorized subject were available.

<img src="data/presidential_doggy_door/train/bo/bo_10.jpg">

## Modeling Challenge & Strategy

Training a deep neural network from scratch on such a small dataset would result in severe overfitting and poor generalization. To address this, I leveraged **transfer learning** using a pretrained convolutional neural network that already learned general visual features related to dogs.

By freezing most of the pretrained layers and fine-tuning the final classification layers, the model was able to generalize effectively despite the limited data. This approach allowed the system to learn discriminative features specific to the authorized subject while retaining robustness to unseen images.

## Model Selection: Pretrained VGG16 Backbone


To enable effective learning from a limited dataset, I selected a **VGG16 convolutional neural network pretrained on ImageNet** as the backbone model.

VGG16 provides a strong feature extractor for visual tasks involving animals, as its convolutional layers have already learned rich hierarchical representations such as edges, textures, shapes, and object-level features. Leveraging these pretrained representations significantly reduces overfitting and improves generalization when labeled data is scarce.

In [2]:
from torchvision.models import vgg16
from torchvision.models import VGG16_Weights

# Load VGG16 pretrained on ImageNet for transfer learning
weights = VGG16_Weights.DEFAULT
vgg_model = vgg16(weights=weights)

The pretrained convolutional layers were retained as a fixed feature extractor, while the classification head was modified in subsequent steps to adapt the network for binary classification (authorized vs. unauthorized).

## Freezing the Pretrained Feature Extractor
To preserve the learned visual representations from ImageNet, all pretrained VGG16 parameters were frozen prior to training. This ensures that only the newly added classification layers are updated during optimization.

Freezing the backbone significantly reduces overfitting when working with small datasets and allows the model to leverage robust, general-purpose visual features.

In [3]:
vgg_model.requires_grad_(False)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

## Custom Classification Head

To adapt the pretrained VGG16 network for the target task, a lightweight classification head was appended to the model.

The final fully connected layer maps the 1000-dimensional ImageNet output space to a single neuron representing a binary decision: authorized (Bo) vs. unauthorized.

In [4]:
N_CLASSES = 1

my_model = nn.Sequential(
    vgg_model,
    nn.Linear(1000, N_CLASSES)
)

my_model.to(device)

Sequential(
  (0): VGG(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=True)
      (16): M

A single-output neuron was used in conjunction with a sigmoid-based loss function to perform binary classification.
Only the parameters of the newly added classification layer were left trainable, ensuring that optimization focused exclusively on task-specific decision boundaries.

If we did want to make the VGG layers trainable, we could take `vgg_model` and set `requires_grad_` to `True`.

But for now, we'd only like to train our new layers, so we will turn training off.

### Compiling the Model

Loss & Optimization
Since this is a binary classification task (Bo vs. not-Bo), I use a single-output neuron with BCEWithLogitsLoss, which is numerically stable and avoids applying a sigmoid inside the model. Optimization is performed using Adam.

In [5]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = Adam(my_model.parameters())
my_model = my_model.to(device)

## Data Augmentation

In [6]:
pre_trans = weights.transforms()

Rather than read from a DataFrame, I will read image files directly and infer the `label` based on the filepath.

In [7]:
DATA_LABELS = ["bo", "not_bo"] 
    
class MyDataset(Dataset):
    def __init__(self, data_dir):
        self.imgs = []
        self.labels = []
        
        for l_idx, label in enumerate(DATA_LABELS):
            data_paths = glob.glob(data_dir + label + '/*.jpg', recursive=True)
            for path in data_paths:
                img = Image.open(path)
                self.imgs.append(pre_trans(img).to(device))
                self.labels.append(torch.tensor(l_idx).to(device).float())


    def __getitem__(self, idx):
        img = self.imgs[idx]
        label = self.labels[idx]
        return img, label

    def __len__(self):
        return len(self.imgs)

### The DataLoaders

Now that I have a custom Dataset class, I create the DataLoaders

In [8]:
n = 32

train_path = "data/presidential_doggy_door/train/"
train_data = MyDataset(train_path)
train_loader = DataLoader(train_data, batch_size=n, shuffle=True)
train_N = len(train_loader.dataset)

valid_path = "data/presidential_doggy_door/valid/"
valid_data = MyDataset(valid_path)
valid_loader = DataLoader(valid_data, batch_size=n)
valid_N = len(valid_loader.dataset)

Applied some data augmentation so the model can have a better chance at recognizing Bo. I have color images, so I  can use ColorJitter.

In [9]:
IMG_WIDTH, IMG_HEIGHT = (224, 224)

random_trans = transforms.Compose([
    transforms.RandomRotation(25),
    transforms.RandomResizedCrop((IMG_WIDTH, IMG_HEIGHT), scale=(.8, 1), ratio=(1, 1)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=.2, contrast=.2, saturation=.2, hue=.2)
])

## The Training Loop

This section defines a lightweight PyTorch training/evaluation loop for binary classification.

- The model outputs **logits** (raw scores).
- We use `BCEWithLogitsLoss`, so we **do not** apply sigmoid during training.
- For accuracy, we convert logits → predicted class using a threshold at 0 (equivalent to sigmoid > 0.5).

In [10]:
def get_batch_accuracy(output, y, N):
    zero_tensor = torch.tensor([0]).to(device)
    pred = torch.gt(output, zero_tensor)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

We also section to print the last set of gradients to show that only our newly added layers are learning.

In [11]:
def train(model, check_grad=False):
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        output = torch.squeeze(model(random_trans(x)))
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
    if check_grad:
        print('Last Gradient:')
        for param in model.parameters():
            print(param.grad)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

The `validate` function

In [12]:
def validate(model):
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            output = torch.squeeze(model(x))

            loss += loss_function(output, y.float()).item()
            accuracy += get_batch_accuracy(output, y, valid_N)
    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

### Model Training

The model was trained for multiple epochs using a frozen feature extractor and a trainable classification head. Training and validation performance were monitored to evaluate convergence and generalization.


In [13]:
epochs = 10

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train(my_model, check_grad=False)
    validate(my_model)

Epoch: 0


OutOfMemoryError: CUDA out of memory. Tried to allocate 392.00 MiB. GPU 

## Results and Evaluation

The model achieved high validation accuracy despite the small dataset size. Leveraging pretrained ImageNet features enabled effective generalization while minimizing overfitting. Validation accuracy stabilized after several epochs, indicating convergence.


## Fine-Tuning Strategy

After training the classification head, the model was fine-tuned by unfreezing the pretrained layers and retraining with a very small learning rate. This allows minor weight adjustments while preserving previously learned representations.


In [ ]:
# Unfreezing the base model
vgg_model.requires_grad_(True)
optimizer = Adam(my_model.parameters(), lr=.000001)

In [ ]:
epochs = 2

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train(my_model, check_grad=False)
    validate(my_model)

## Model Inference and Prediction Analysis

To evaluate real-world behavior, the trained model was tested on unseen validation images. Predictions were generated directly from raw image inputs using the same preprocessing pipeline applied during training.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def show_image(image_path):
    image = mpimg.imread(image_path)
    plt.imshow(image)

In [ ]:
def make_prediction(file_path):
    show_image(file_path)
    image = Image.open(file_path)
    image = pre_trans(image).to(device)
    image = image.unsqueeze(0)
    output = my_model(image)
    prediction = output.item()
    return prediction

In [ ]:
make_prediction('data/presidential_doggy_door/valid/bo/bo_20.jpg')

In [ ]:
make_prediction('data/presidential_doggy_door/valid/not_bo/121.jpg')

Negative number prediction means that it is Bo, and a positive number prediction means it is something else. We can use this information to have our doggy door only let Bo in.

## Bo's Doggy Door

In [ ]:
def presidential_doggy_door(image_path):
    pred = make_prediction(image_path)
    if pred < 0:
        print("It's Bo. Let him in.")
    else:
        print("That's not Bo. Stay out.")

In [ ]:
presidential_doggy_door('data/presidential_doggy_door/valid/not_bo/131.jpg')

In [ ]:
presidential_doggy_door('data/presidential_doggy_door/valid/bo/bo_29.jpg')

## Project Summary

This project demonstrates the application of transfer learning for binary image classification using a pretrained VGG16 model. By freezing the feature extractor and training a lightweight classification head, strong performance was achieved despite a limited dataset.

Fine-tuning with a reduced learning rate further improved model stability while preserving pretrained representations. This workflow highlights an effective approach for adapting large vision models to small, domain-specific datasets.

## Conclusion

This project demonstrates how pretrained convolutional neural networks can be effectively adapted to new classification tasks with limited labeled data. The combination of frozen feature extraction, targeted fine-tuning, and consistent preprocessing enables strong generalization while maintaining training efficiency.